In [2]:
# =========================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# =========================================
print("Cargando librerías...")
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from flashtext import KeywordProcessor

# Configuraciones de display (opcional pero útil)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Librerías cargadas.")

# =========================================
# 2. CARGA DE DATOS GUARDADOS
# =========================================
print("\nCargando archivos guardados...")
try:
    # Cargar nuestro DataFrame principal (el de 35k filas)
    df_master = pd.read_pickle('df_master_clean.pkl')
    
    # Cargar nuestro "diccionario" de habilidades de O*NET
    df_onet_tech_skills = pd.read_excel('datasets/ONET/Technology Skills.xlsx')
    
    print(f"Éxito: 'df_master_clean.pkl' cargado ({df_master.shape[0]} filas).")
    print(f"Éxito: 'Technology Skills.xlsx' cargado.")

except FileNotFoundError:
    print("--- ¡ERROR! ---")
    print("Asegúrate de que 'df_master_clean.pkl' y 'Technology Skills.xlsx' estén en la misma carpeta.")
except Exception as e:
    print(f"Ocurrió un error al cargar: {e}")


# =========================================
# 3. RE-CREACIÓN DE VARIABLES DE ANÁLISIS
# (Esto re-genera 'corr_matrix')
# =========================================
print("\nRe-creando variables de análisis (corr_matrix)...")
try:
    # --- 3a. Re-ejecutar el escaneo FlashText ---
    skill_list = df_onet_tech_skills['Example'].dropna().astype(str).str.lower().unique().tolist()
    keyword_processor = KeywordProcessor()
    keyword_processor.add_keywords_from_list(skill_list)
    df_master['description'] = df_master['description'].fillna('')
    df_master['granular_skills'] = df_master['description'].apply(lambda x: keyword_processor.extract_keywords(x.lower()))

    # --- 3b. Re-calcular la demanda (Top 20) ---
    df_master['granular_skills_unique'] = df_master['granular_skills'].apply(lambda skills_list: list(set(skills_list)))
    df_exploded = df_master.explode('granular_skills_unique')
    granular_demand = df_exploded['granular_skills_unique'].value_counts()
    top_20_skill_names = granular_demand.head(20).index.tolist()

    # --- 3c. Re-crear el DataFrame para el heatmap ---
    df_heatmap = pd.DataFrame()
    for skill in top_20_skill_names:
        df_heatmap[skill] = df_master['granular_skills_unique'].apply(
            lambda skills_list: 1 if skill in skills_list else 0
        )
    
    # --- 3d. ¡La variable clave que necesitamos! ---
    corr_matrix = df_heatmap.corr()
    
    print("¡Éxito! La variable 'corr_matrix' ha sido re-creada y está lista para usarse.")
    
except Exception as e:
    print(f"Ocurrió un error al re-crear las variables: {e}")

Cargando librerías...
Librerías cargadas.

Cargando archivos guardados...
Éxito: 'df_master_clean.pkl' cargado (35085 filas).
Éxito: 'Technology Skills.xlsx' cargado.

Re-creando variables de análisis (corr_matrix)...
¡Éxito! La variable 'corr_matrix' ha sido re-creada y está lista para usarse.


In [3]:
# =========================================
# 4. HERRAMIENTA DE RECOMENDACIÓN
# =========================================
try:
    def get_recommendations(skill_name, corr_matrix):
        """
        Consulta nuestra matriz de correlación para encontrar las habilidades
        más relacionadas con una habilidad dada.
        """
        print(f"--- Recomendaciones para alguien que sabe: {skill_name} ---")
        
        if skill_name not in corr_matrix:
            return f"Error: La habilidad '{skill_name}' no está en nuestro Top 20."

        skill_correlations = corr_matrix[skill_name]
        sorted_correlations = skill_correlations.sort_values(ascending=False)
        recommended_skills = sorted_correlations.drop(skill_name)
        
        return recommended_skills.head(3)

    # --- ¡Prueba la herramienta! ---
    
    # Ejemplo 1: ¿Qué aprender si sé 'docker'?
    recs_docker = get_recommendations('docker', corr_matrix)
    print(recs_docker)

    print("\n" + "="*30 + "\n")

    # Ejemplo 2: ¿Qué aprender si sé 'python'?
    recs_python = get_recommendations('python', corr_matrix)
    print(recs_python)

except Exception as e:
    print(f"Ocurrió un error: {e}")

--- Recomendaciones para alguien que sabe: docker ---
kubernetes    0.660173
nosql         0.257262
python        0.229223
Name: docker, dtype: float64


--- Recomendaciones para alguien que sabe: python ---
c++                      0.318507
programming languages    0.313460
docker                   0.229223
Name: python, dtype: float64


In [4]:
# =========================================
# 4. HERRAMIENTA DE RECOMENDACIÓN (Versión 2.0 - Co-ocurrencia)
# =========================================

# (Necesitamos esta lista de ruido que identificamos antes)
noise_skills = ['analyze', 'go', 'c', 'r', 'reduce', 'google', 'linkedin', 
                'programming languages', 'e-verify', 'c#', 'c++']

def get_recommendations_v2(skill_name, df):
    """
    Encuentra las habilidades más co-ocurrentes para una habilidad dada.
    """
    print(f"--- Recomendaciones (Co-ocurrencia) para: {skill_name} ---")
    
    # 1. Filtrar el DataFrame para obtener solo trabajos que mencionan la habilidad
    # .apply() es un poco lento, pero robusto.
    jobs_with_skill = df[df['granular_skills_unique'].apply(
        lambda skills_list: skill_name in skills_list
    )]
    
    print(f"Se encontraron {len(jobs_with_skill)} trabajos que mencionan '{skill_name}'.")

    # 2. "Explotar" solo ese subconjunto de trabajos
    df_exploded_subset = jobs_with_skill.explode('granular_skills_unique')
    
    # 3. Contar la frecuencia de todas las otras habilidades
    skill_counts = df_exploded_subset['granular_skills_unique'].value_counts()
    
    # 4. Limpiar la lista:
    # 4a. Eliminar la habilidad de búsqueda (ej. 'python')
    skill_counts = skill_counts.drop(skill_name)
    
    # 4b. Eliminar otras habilidades de "ruido"
    skill_counts = skill_counts.drop(labels=noise_skills, errors='ignore')
    
    # 5. Devolver las 5 mejores recomendaciones
    return skill_counts.head(5)

# --- ¡Vamos a Probar la Nueva Versión! ---
try:
    # (Asegúrate de que 'df_master' esté cargado y tenga 'granular_skills_unique')
    
    # Ejemplo 1: ¿Qué aprender si sé 'docker'?
    recs_docker = get_recommendations_v2('docker', df_master)
    print(recs_docker)

    print("\n" + "="*30 + "\n")

    # Ejemplo 2: ¿Qué aprender si sé 'python'?
    recs_python = get_recommendations_v2('python', df_master)
    print(recs_python)

except Exception as e:
    print(f"Ocurrió un error: {e}")

--- Recomendaciones (Co-ocurrencia) para: docker ---
Se encontraron 969 trabajos que mencionan 'docker'.
granular_skills_unique
kubernetes    679
python        527
git           254
linux         248
javascript    248
Name: count, dtype: int64


--- Recomendaciones (Co-ocurrencia) para: python ---
Se encontraron 3968 trabajos que mencionan 'python'.
granular_skills_unique
javascript    741
linux         637
kubernetes    538
docker        527
git           440
Name: count, dtype: int64
